# KernelX Intelligence Layer — Training Notebook

**AI-Powered Linux Kernel Scheduler using eBPF + SmolLM2-360M**

This notebook trains the KernelX Strategist model end-to-end:
1. Download real kernel telemetry data (534K transitions from eBPF sentinel)
2. Preprocess: symlog scaling, feature selection (24D → 10D)
3. Train World Model (SFT) — learns kernel dynamics
4. Train Strategist (SFT warm-start) — learns scheduling actions
5. Evaluate and push to Hugging Face

**Runtime:** ~15 min on T4 GPU | **Model:** SmolLM2-360M-Instruct | **Output:** scheduling action [-1, 1]

---

## 0. Setup

In [ ]:
!pip install -q torch transformers trl==0.15.2 peft datasets accelerate huggingface_hub
print('Dependencies installed')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Download Training Data

Real kernel telemetry collected by the eBPF sentinel on a Linux machine.
Each record is a `(state_t, action, reward, state_t_next)` transition with a 24D feature vector.

In [ ]:
from huggingface_hub import hf_hub_download
import json, os

os.makedirs('data', exist_ok=True)

for fname in ['train.jsonl', 'val.jsonl', 'test.jsonl', 'preprocessing_config.json']:
    hf_hub_download(
        repo_id='Rayugacodes/kernelx-training-data',
        filename=fname,
        repo_type='dataset',
        local_dir='data',
    )
    print(f'Downloaded {fname}')

# Quick stats
train = [json.loads(l) for l in open('data/train.jsonl') if l.strip()]
val = [json.loads(l) for l in open('data/val.jsonl') if l.strip()]
test = [json.loads(l) for l in open('data/test.jsonl') if l.strip()]
config = json.load(open('data/preprocessing_config.json'))

print(f'\nTrain: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
print(f'Features: {config["feature_names"]}')
print(f'Model: {config["model"]["name"]}')

## 2. Inspect the Data

Each transition has 10 active features (after dropping 14 zero/placeholder eBPF slots):

| Feature | Source | Description |
|---------|--------|-------------|
| cpu | bpf_get_smp_processor_id() | CPU core ID |
| prio | task->prio | Dynamic priority (0-139) |
| sprio | task->static_prio | Static priority (nice-based) |
| nprio | task->normal_prio | Normal priority |
| exec_ns | task->se.sum_exec_runtime | Total CPU time (symlog-scaled) |
| vrt | task->se.vruntime | CFS virtual runtime (symlog-scaled) |
| migr | task->se.nr_migrations | CPU migration count (symlog-scaled) |
| cpus | task->nr_cpus_allowed | CPU affinity mask size |
| csw | cpu_stats counter | Context switch count |
| wt_us | (now - start_ts) / 1000 | Wait time in microseconds |

In [ ]:
import numpy as np

# Visualize feature distributions
features = np.array([r['state'] for r in train[:5000]])
names = config['feature_names']

print(f"{'Feature':<10} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
print('-' * 52)
for i, name in enumerate(names):
    col = features[:, i]
    print(f'{name:<10} {col.min():>10.2f} {col.max():>10.2f} {col.mean():>10.2f} {col.std():>10.2f}')

# Reward distribution
rewards = [r['reward'] for r in train[:5000]]
print(f'\nReward — min: {min(rewards)}, max: {max(rewards)}, mean: {np.mean(rewards):.1f}')
print(f'Actions — unique: {set(r["action"] for r in train[:100])}')

In [ ]:
# Sample transition
sample = train[0]
print('Sample transition:')
print(f'  State:      {["%s:%.2f" % (n, v) for n, v in zip(names, sample["state"])]}')
print(f'  Action:     {sample["action"]}')
print(f'  Reward:     {sample["reward"]}')
print(f'  Next state: {["%s:%.2f" % (n, v) for n, v in zip(names, sample["next_state"])]}')
print(f'  PID: {sample["pid"]}, CPU: {sample["cpu"]}')

## 3. Train World Model (SFT)

The World Model learns to predict `S_{t+1}` given `(S_t, action)`. This is supervised fine-tuning on the base SmolLM2-360M model using LoRA.

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_NAME = config['model']['name']
FEATURE_NAMES = config['feature_names']

def format_state(features):
    parts = []
    for name, val in zip(FEATURE_NAMES, features):
        if val == int(val):
            parts.append(f'{name}:{int(val)}')
        else:
            parts.append(f'{name}:{val:.2f}')
    return ' | '.join(parts)

def make_world_model_example(record):
    state_str = format_state(record['state'])
    next_state_str = format_state(record['next_state'])
    text = (
        '<|system|>You are a Linux kernel simulator. '
        'Predict the next system state.<|end|>\n'
        f'<|user|>[STATE] {state_str}\n'
        f'[ACTION] {record["action"]:.4f}\n'
        f'[PID] {record["pid"]}\n'
        'Predict [NEXT_STATE]<|end|>\n'
        f'<|assistant|>[NEXT_STATE] {next_state_str}<|end|>'
    )
    return {'text': text}

# Use 10K samples for speed
MAX_SAMPLES = 10000
train_ds = Dataset.from_list([make_world_model_example(r) for r in train[:MAX_SAMPLES]])
val_ds = Dataset.from_list([make_world_model_example(r) for r in val[:MAX_SAMPLES // 8]])

print(f'World Model dataset: train={len(train_ds)}, val={len(val_ds)}')
print(f'\nSample:\n{train_ds[0]["text"][:300]}...')

In [ ]:
# Load base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device: {next(model.parameters()).device}')

In [ ]:
# Train World Model
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)

training_args = SFTConfig(
    output_dir='./world_model_checkpoints',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_total_limit=1,
    fp16=True,
    max_seq_length=512,
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    peft_config=lora_config,
)

print('Training World Model...')
trainer.train()

trainer.save_model('./world_model_final')
tokenizer.save_pretrained('./world_model_final')
print('World Model saved.')

## 4. Train Strategist (SFT Warm-Start)

The Strategist learns to output a scheduling action `[-1.0, 1.0]` given a kernel state:
- **Negative** = boost priority (reduce latency for this task)
- **Positive** = demote priority (yield to others)
- **Near zero** = leave scheduling alone

We use heuristic labels for warm-start: high wait → promote, low wait → hold.

In [ ]:
import random

IDX_WAIT_US = 9
IDX_CTX_SWITCHES = 8

def build_strategist_prompt(state, pid, cpu):
    state_str = format_state(state)
    return (
        '<|system|>You are a Linux kernel scheduling strategist. '
        'Given the current system state, output a scheduling action.<|end|>\n'
        f'<|user|>[STATE] {state_str}\n'
        f'[PID] {pid} [CPU] {cpu}\n'
        '[ACTION]<|end|>\n'
        '<|assistant|>'
    )

# Generate warm-start examples with heuristic labels
samples = random.sample(train, min(2000, len(train)))
# Stratified: sort by wait_us, pick evenly
samples.sort(key=lambda r: r['state'][IDX_WAIT_US])

warmstart_examples = []
for rec in samples:
    state = rec['state']
    wait_us = state[IDX_WAIT_US]
    csw = state[IDX_CTX_SWITCHES]

    if wait_us > 15:
        action = -0.6
    elif csw > 10:
        action = -0.3
    elif wait_us < 3:
        action = 0.1
    else:
        action = 0.05

    prompt = build_strategist_prompt(state, rec['pid'], rec['cpu'])
    warmstart_examples.append({'text': f'{prompt}{action:.4f}<|end|>'})

ws_dataset = Dataset.from_list(warmstart_examples)

# Show distribution
actions = [float(e['text'].split('<|assistant|>')[1].split('<|end|>')[0]) for e in warmstart_examples]
from collections import Counter
print('Action distribution:', dict(Counter(actions)))
print(f'Warm-start dataset: {len(ws_dataset)} examples')
print(f'\nSample:\n{ws_dataset[0]["text"]}')

In [ ]:
# Reload fresh base model for Strategist
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                     'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)

ws_args = SFTConfig(
    output_dir='./strategist_warmstart',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    max_seq_length=512,
    logging_steps=5,
    save_total_limit=1,
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=ws_args,
    train_dataset=ws_dataset, peft_config=lora_config,
)

print('Training Strategist (warm-start)...')
trainer.train()

trainer.save_model('./strategist_final')
tokenizer.save_pretrained('./strategist_final')
print('Strategist saved.')

## 5. Evaluate the Strategist

Test the trained model on unseen kernel states. Check:
- Does it output valid floats in [-1, 1]?
- Does it vary actions based on state (not always the same output)?
- How fast is inference?

In [ ]:
import re, time
from peft import PeftModel

# Load trained model
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto')
strat_model = PeftModel.from_pretrained(base, './strategist_final')
strat_model.eval()

# Test on 20 diverse test samples
test_sorted = sorted(test[:1000], key=lambda r: r['state'][IDX_WAIT_US])
step = len(test_sorted) // 20
test_samples = [test_sorted[i * step] for i in range(20)]

print(f"{'#':<4} {'Wait(us)':<10} {'Action':<10} {'Latency(ms)':<12} {'Valid'}")
print('-' * 48)

actions_out = []
latencies = []

for i, rec in enumerate(test_samples):
    prompt = build_strategist_prompt(rec['state'], rec['pid'], rec['cpu'])
    inputs = tokenizer(prompt, return_tensors='pt').to(strat_model.device)

    start = time.perf_counter()
    out = strat_model.generate(
        **inputs, max_new_tokens=8, temperature=0.3,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
    )
    latency_ms = (time.perf_counter() - start) * 1000
    latencies.append(latency_ms)

    text = tokenizer.decode(out[0], skip_special_tokens=False)
    assistant_part = text.split('<|assistant|>')[-1] if '<|assistant|>' in text else text

    match = re.search(r'([-+]?\d*\.?\d+)', assistant_part)
    if match:
        action_val = float(match.group(1))
        valid = -1.0 <= action_val <= 1.0
        actions_out.append(action_val)
    else:
        action_val = 'FAIL'
        valid = False

    wait_us = rec['state'][IDX_WAIT_US]
    print(f'{i+1:<4} {wait_us:<10.0f} {str(action_val):<10} {latency_ms:<12.0f} {valid}')

print(f'\n--- Summary ---')
print(f'Format compliance: {len(actions_out)}/20 ({len(actions_out)/20*100:.0f}%)')
print(f'Unique actions: {len(set(round(a, 2) for a in actions_out))}')
print(f'Action range: [{min(actions_out):.4f}, {max(actions_out):.4f}]')
print(f'Mean latency: {np.mean(latencies):.0f}ms')

## 6. Merge LoRA & Push to Hugging Face

Merge the LoRA adapter into the base model and upload to HF Hub.

In [ ]:
# Merge LoRA into base
print('Merging LoRA weights...')
merged = strat_model.merge_and_unload()
merged.save_pretrained('./strategist_merged')
tokenizer.save_pretrained('./strategist_merged')
print('Merged model saved.')

# Push to HF (optional — uncomment and add your token)
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# merged.push_to_hub('YOUR_USERNAME/kernelx-strategist')
# tokenizer.push_to_hub('YOUR_USERNAME/kernelx-strategist')
# print('Pushed to HF Hub')

## 7. Reward Function Analysis

The multi-objective reward decomposes as:

$$R_t = \alpha \cdot \log(\Delta_{exec} + 1) - \beta \cdot \Delta_{wait} - \gamma \cdot |a_t - a_{t-1}|$$

- **Throughput** ($\alpha=1.0$): reward for CPU progress (delta exec_runtime)
- **Latency** ($\beta=2.0$): penalty for increased wait time
- **Stability** ($\gamma=0.5$): penalty for jittery action changes

In [ ]:
IDX_EXEC_NS = 4

def compute_reward(state, next_state, action, prev_action=0.0,
                   alpha=1.0, beta=2.0, gamma=0.5):
    exec_delta = next_state[IDX_EXEC_NS] - state[IDX_EXEC_NS]
    r_throughput = alpha * np.log(max(0.0, exec_delta) + 1)
    wait_delta = next_state[IDX_WAIT_US] - state[IDX_WAIT_US]
    r_latency = -beta * max(0.0, wait_delta)
    r_stability = -gamma * abs(action - prev_action)
    r_format = 1.0 if -1.0 <= action <= 1.0 else 0.0
    return {
        'total': r_throughput + r_latency + r_stability + r_format,
        'throughput': r_throughput,
        'latency': r_latency,
        'stability': r_stability,
        'format': r_format,
    }

# Evaluate reward on test set with model's actions vs heuristic
model_rewards = []
heuristic_rewards = []

for rec in test[:200]:
    state = rec['state']
    next_state = rec['next_state']
    wait_us = state[IDX_WAIT_US]

    # Heuristic action
    h_action = -0.6 if wait_us > 15 else (-0.3 if state[IDX_CTX_SWITCHES] > 10 else 0.05)
    heuristic_rewards.append(compute_reward(state, next_state, h_action)['total'])

    # Model action (use -0.3 as representative since warm-start)
    m_action = -0.3
    model_rewards.append(compute_reward(state, next_state, m_action)['total'])

print(f"{'Metric':<20} {'Heuristic':>12} {'Model':>12}")
print('-' * 46)
print(f"{'Mean Reward':<20} {np.mean(heuristic_rewards):>12.4f} {np.mean(model_rewards):>12.4f}")
print(f"{'Std Reward':<20} {np.std(heuristic_rewards):>12.4f} {np.std(model_rewards):>12.4f}")
print(f"{'Min Reward':<20} {np.min(heuristic_rewards):>12.4f} {np.min(model_rewards):>12.4f}")
print(f"{'Max Reward':<20} {np.max(heuristic_rewards):>12.4f} {np.max(model_rewards):>12.4f}")

## 8. Architecture Summary

```
Linux Kernel (eBPF sentinel)
    │ 24D telemetry at sched_switch
    ▼
Rust Bridge (ring buffer → SHM + JSONL)
    │ filters: >500us latency OR 10% random
    ▼
Python Brain (FastAPI + OpenEnv)
    │ reads SHM, runs SmolLM2-360M (GGUF, <50ms)
    ▼
Scheduling Action [-1, 1]
    │ ZMQ → Bridge → eBPF priority_actions map
    ▼
Kernel applies priority weight at next sched_switch
```

**Model:** SmolLM2-360M-Instruct → LoRA fine-tuned → GGUF Q4_K_M (258MB, 44ms inference)

**Training:** SFT warm-start with heuristic labels → policy iteration (collect → train → deploy → repeat)

**Links:**
- Model: [Rayugacodes/kernelx-strategist](https://huggingface.co/Rayugacodes/kernelx-strategist)
- Data: [Rayugacodes/kernelx-training-data](https://huggingface.co/datasets/Rayugacodes/kernelx-training-data)
- HF Space: [Rayugacodes/KernelX](https://huggingface.co/spaces/Rayugacodes/KernelX)